In [1]:
!pip install tensorflow

  Using cached protobuf-4.25.5-cp37-abi3-macosx_10_9_universal2.whl.metadata (541 bytes)
Using cached protobuf-4.25.5-cp37-abi3-macosx_10_9_universal2.whl (394 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.1
    Uninstalling protobuf-5.29.1:
      Successfully uninstalled protobuf-5.29.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.28.2 requires protobuf<6.0,>=5.0, but you have protobuf 4.25.5 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.16.2 which is incompatible.


## Import necessary libraries


In [2]:

import os
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertModel, AdamW, BertTokenizerFast, BertForTokenClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder

from tqdm import tqdm

## Installing necessary libraries
import tensorflow as tf
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import gensim
from gensim.models import Word2Vec, Doc2Vec
from transformers import BertTokenizer, BertModel
import torch
from gensim.models.doc2vec import TaggedDocument
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


from tensorflow.keras import layers, models
from tensorflow.keras.layers import Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed, Bidirectional, Input, Dropout



import time
import unicodedata



/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!python -m spacy download en_core_web_md

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
nlp = spacy.load('en_core_web_md')
pd.set_option('display.max_colwidth',100)


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 30.6 MB/s eta 0:00:00a 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


[nltk_data] Downloading package punkt to /Users/monilshah/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/monilshah/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/monilshah/nltk_data...


## Getting the data

In [4]:
training_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/02_wip_data/relations_train_data.json'
test_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/01_Input/02_wip_data/relations_test_data.json'

In [5]:
# Load the JSON file
with open(training_data_path, 'r') as file:
    all_data = json.load(file)

# Example structure of each data entry
# {
#     "news_line": "Sentence text here.",
#     "triples": [
#         {"subject": "Entity1", "object": "Entity2", "relation": "RelationType"}
#     ]
# }



In [6]:
data = all_data[0]

In [7]:
data[0]

{'news_line': 'NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns about his health on Monday, saying his weight loss was caused by a hormone imbalance that is relatively simple to treat.',
 'triples': [{'subject': 'Apple Inc',
   'object': 'Steve Jobs',
   'relation': 'founded_by'},
  {'subject': 'Apple Inc',
   'object': 'Steve Jobs',
   'relation': 'chief_executive_officer'}]}

In [8]:
# Expand each triple into a row
rows = []
for item in data:  # Loop through each item in the list
    for triple in item['triples']:
        rows.append({
            'news_line': item['news_line'],
            'subject': triple['subject'],
            'object': triple['object'],
            'relation': triple['relation']
        })

# Create the DataFrame
df = pd.DataFrame(rows)
print(df)


                                                                                                news_line  \
0     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
1     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
2     Last week, Citigroup Inc's ( C.N ) Chief Executive Vikram Pandit said that he, Chairman Win Bisc...   
3     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
4     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
...                                                                                                   ...   
8074  In particular, he said leases of used A330-200 aircraft from Airbus Group SE (Xetra: A1XBMK - ne...   
8075  The company is an omnipresent in households the world over thanks to its operations across multi...   
8076               

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['relation'] = le.fit_transform(df['relation'])

In [10]:
def get_filtered_entities(text):
    
    text = text.replace("Inc ", "Inc. ")
    
    doc = nlp(text)
    # Collect only entities that are either a person or an organization
    entities = set([ent.text for ent in doc.ents if ent.label_ in {"PERSON", "ORG"}])
    return entities

# Apply the function and convert the set to a comma-separated string
df['entities'] = df['news_line'].apply(lambda x: ', '.join(get_filtered_entities(x)))

print(df)


                                                                                                news_line  \
0     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
1     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
2     Last week, Citigroup Inc's ( C.N ) Chief Executive Vikram Pandit said that he, Chairman Win Bisc...   
3     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
4     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
...                                                                                                   ...   
8074  In particular, he said leases of used A330-200 aircraft from Airbus Group SE (Xetra: A1XBMK - ne...   
8075  The company is an omnipresent in households the world over thanks to its operations across multi...   
8076               

In [11]:
# Split data into training and validation sets
#df_test = df[7000:]
#df = df[:7000]

In [12]:
def prepare_relation_data(df, maxlen=None):
    # Add markers to the news line
    def add_markers(row):
        news_line = row['news_line']
        subject = row['subject']
        object_ = row['object']
        
        news_line = news_line.replace(subject, f"[SUBJECT] {subject} [SUBJECT_END]")
        news_line = news_line.replace(object_, f"[OBJECT] {object_} [OBJECT_END]")
        return news_line
    
    df['marked_news_line'] = df.apply(add_markers, axis=1)
    
    # Tokenize and convert to sequences
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(df['marked_news_line'])
    
    X_seq = tokenizer.texts_to_sequences(df['marked_news_line'])
    y = df['relation']  # Assuming `relation` is already encoded as integers
    
    # Pad sequences
    if maxlen is None:
        maxlen = max(len(seq) for seq in X_seq)
    X_pad = pad_sequences(X_seq, padding='post', maxlen=maxlen)
    
    return X_pad, np.array(y), tokenizer


In [13]:

def build_relation_model(vocab_size, embedding_dim, maxlen, num_relations):
    # Input layer
    input_layer = Input(shape=(maxlen,))
    
    # Embedding layer
    embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen)(input_layer)
    
    # LSTM layer
    lstm_layer = Bidirectional(LSTM(units=128, return_sequences=False))(embedding_layer)
    
    # Fully connected layer with dropout
    dense_layer_1 = Dense(256, activation='relu')(lstm_layer)
    dropout_layer_1 = Dropout(0.3)(dense_layer_1)  # Dropout to prevent overfitting
    dense_layer_2 = Dense(128, activation='relu')(dropout_layer_1)
    dropout_layer_2 = Dropout(0.3)(dense_layer_2)  # Dropout to prevent overfitting
    dense_layer_3 = Dense(128, activation='relu')(dropout_layer_2)
    dropout_layer_3 = Dropout(0.3)(dense_layer_3)  # Dropout to prevent overfitting
 

    # Output layer for relation classification
    output_layer = Dense(num_relations, activation='softmax')(dropout_layer_3)
    
    # Build and compile the model
    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


In [14]:

# Prepare callbacks
checkpoint = ModelCheckpoint(
    "best_model.keras",  # Filepath to save the best model
    monitor="val_loss",  # Metric to monitor
    save_best_only=True,  # Only save the best model
    mode="min",  # Lower validation loss is better
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",  # Metric to monitor
    patience=10,  # Stop training after 5 epochs with no improvement
    mode="min",  # Lower validation loss is better
    verbose=1
)

# Prepare data
X_pad, y, tokenizer = prepare_relation_data(df, maxlen=64)

# Split data into training and validation sets
#X_train, X_val, y_train, y_val = train_test_split(X_pad, y, test_size=0.2, random_state=42)

# Split data into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X_pad, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Build model
vocab_size = len(tokenizer.word_index) + 1  # +1 for padding token
embedding_dim = 64
num_relations = len(set(y))  # Number of relation classes

model = build_relation_model(vocab_size, embedding_dim, maxlen=64, num_relations=num_relations)

# Train the model with callbacks
history = model.fit(
    X_train, 
    y_train, 
    epochs=100, 
    batch_size=4, 
    validation_data=(X_val, y_val), 
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


Epoch 1/100


/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1414/1414 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.1627 - loss: 2.9365
Epoch 1: val_loss improved from inf to 2.51506, saving model to best_model.keras
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 35s 24ms/step - accuracy: 0.1627 - loss: 2.9364 - val_accuracy: 0.2368 - val_loss: 2.5151
Epoch 2/100
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.2393 - loss: 2.4030
Epoch 2: val_loss improved from 2.51506 to 2.16578, saving model to best_model.keras
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 36s 25ms/step - accuracy: 0.2393 - loss: 2.4029 - val_accuracy: 0.3003 - val_loss: 2.1658
Epoch 3/100
1412/1414 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.3560 - loss: 1.9079
Epoch 3: val_loss improved from 2.16578 to 2.00701, saving model to best_model.keras
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 174s 123ms/step - accuracy: 0.3560 - loss: 1.9079 - val_accuracy: 0.3482 - val_loss: 2.0070
Epoch 4/100
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4529 - loss: 1.5995
Epoch 4: val_loss improved from 2.0

In [17]:
from tensorflow.keras.models import load_model

# Load the best saved model
best_model = load_model("best_model.keras")

# Evaluate on validation data
val_loss, val_accuracy = best_model.evaluate(X_val, y_val, verbose=1)
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")



38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4543 - loss: 1.8033
Validation Loss: 1.8428, Validation Accuracy: 0.4620


In [18]:
from sklearn.metrics import accuracy_score

# Predict on test set
y_pred_probs = best_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)  # Convert probabilities to class indices

# Compare with y_test
print("Accuracy on Test Data:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Accuracy on Test Data: 0.4636963696369637

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        12
           1       0.00      0.00      0.00         6
           2       0.83      0.71      0.77        21
           3       0.29      0.48      0.36        27
           4       0.00      0.00      0.00        14
           5       0.38      0.43      0.40         7
           6       0.00      0.00      0.00        34
           7       0.00      0.00      0.00         7
           8       0.00      0.00      0.00        13
           9       0.00      0.00      0.00        12
          10       0.80      0.64      0.71        69
          11       0.11      0.03      0.05        30
          12       0.36      0.56      0.44        97
          13       0.67      0.70      0.69       195
          14       0.67      0.33      0.44        12
          15       0.00     

/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i